# Fetch Risk-Free Rate Data

This notebook fetches daily US Treasury yields (risk-free rate) from FRED (Federal Reserve Economic Data) for the period 2018-12-31 to 2025-10-31.

In [2]:
import pandas as pd
import pandas_datareader as pdr
from datetime import datetime
import os

In [3]:
# Define date range
start_date = '2018-12-31'
end_date = '2025-10-31'

# FRED series for Treasury yields
# DGS3MO: 3-Month Treasury (commonly used as risk-free rate)
# DGS1: 1-Year Treasury
# DGS10: 10-Year Treasury

treasury_series = {
    'DGS3MO': '3-Month Treasury',
    'DGS1': '1-Year Treasury', 
    'DGS10': '10-Year Treasury'
}

print(f"Fetching Treasury yields from {start_date} to {end_date}")

Fetching Treasury yields from 2018-12-31 to 2025-10-31


In [4]:
# Fetch data from FRED
risk_free_data = {}

for series_id, series_name in treasury_series.items():
    try:
        data = pdr.DataReader(series_id, 'fred', start_date, end_date)
        risk_free_data[series_id] = data[series_id]
        print(f"✓ Fetched {series_name} ({series_id}): {len(data)} observations")
    except Exception as e:
        print(f"✗ Error fetching {series_name}: {e}")

# Combine into DataFrame
rf_df = pd.DataFrame(risk_free_data)
rf_df.index.name = 'Date'

print(f"\nTotal observations: {len(rf_df)}")
rf_df.head()

✓ Fetched 3-Month Treasury (DGS3MO): 1785 observations
✓ Fetched 1-Year Treasury (DGS1): 1785 observations
✓ Fetched 10-Year Treasury (DGS10): 1785 observations

Total observations: 1785
✓ Fetched 10-Year Treasury (DGS10): 1785 observations

Total observations: 1785


,DGS3MO,DGS1,DGS10
Date,,,
2018-12-31,2.45,2.63,2.69
2019-01-01,NaN,NaN,NaN
2019-01-02,2.42,2.60,2.66
2019-01-03,2.41,2.50,2.56
2019-01-04,2.42,2.57,2.67


In [5]:
# Clean data - convert yields from percentage to decimal and forward fill missing values
rf_df_clean = rf_df.copy()

# FRED reports yields in percentage form (e.g., 4.5 for 4.5%)
# Convert to decimal (e.g., 0.045 for 4.5%)
rf_df_clean = rf_df_clean / 100

# Forward fill missing values (weekends/holidays are excluded in FRED data)
rf_df_clean = rf_df_clean.ffill()

# Rename columns for clarity
rf_df_clean.columns = ['RF_3M', 'RF_1Y', 'RF_10Y']

print("Data summary (in decimal form):")
print(rf_df_clean.describe())
print(f"\nMissing values:\n{rf_df_clean.isna().sum()}")

Data summary (in decimal form):
             RF_3M        RF_1Y       RF_10Y
count  1785.000000  1785.000000  1785.000000
mean      0.027269     0.026948     0.028113
std       0.021336     0.019576     0.013421
min       0.000000     0.000400     0.005200
25%       0.001400     0.002600     0.015900
50%       0.024400     0.026000     0.028500
75%       0.047200     0.045900     0.041400
max       0.056300     0.054900     0.049800

Missing values:
RF_3M     0
RF_1Y     0
RF_10Y    0
dtype: int64


In [6]:
# Save to CSV in the data folder
output_path = '../data/risk_free_rates.csv'

# Ensure the data directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save with date index
rf_df_clean.to_csv(output_path)

print(f"✓ Saved risk-free rate data to: {output_path}")
print(f"  Date range: {rf_df_clean.index[0].strftime('%Y-%m-%d')} to {rf_df_clean.index[-1].strftime('%Y-%m-%d')}")
print(f"  Rows: {len(rf_df_clean)}")
print(f"  Columns: {list(rf_df_clean.columns)}")

✓ Saved risk-free rate data to: ../data/risk_free_rates.csv
  Date range: 2018-12-31 to 2025-10-31
  Rows: 1785
  Columns: ['RF_3M', 'RF_1Y', 'RF_10Y']


In [7]:
# Verify saved data
verify_df = pd.read_csv(output_path, index_col='Date', parse_dates=True)
print("Verification - First 5 rows of saved data:")
verify_df.head()

Verification - First 5 rows of saved data:


,RF_3M,RF_1Y,RF_10Y
Date,,,
2018-12-31,0.0245,0.0263,0.0269
2019-01-01,0.0245,0.0263,0.0269
2019-01-02,0.0242,0.0260,0.0266
2019-01-03,0.0241,0.0250,0.0256
2019-01-04,0.0242,0.0257,0.0267
